# Rishton Ke Mod (`रिश्तों के मोड़`) — Google Colab Live-Action Video Assembler

This notebook connects to your Google Drive folder `Rishton_Ke_Mod/2026-11-11/`, loads the `15` photorealistic character keyframes (`shot_01a.jpg`–`shot_05c.jpg`) and `5` Hindi multi-voice stems (`hi_01_raw.mp3`–`hi_05_raw.mp3`), lets you either **generate continuous moving + lip-synced `.mp4` clips** via your API key (e.g., Google Gemini/Veo or Fal.ai Kling/Wan) **or drop `.mp4` clips** into `clips/`, and renders the final 1080p 16:9 video + 9:16 Shorts using the repository's `studio/daily_pipeline.py`.

In [ ]:
# 1. Mount Google Drive and clone the studio repository
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
from pathlib import Path

REPO_DIR = Path('/content/contentcreation')
if not REPO_DIR.exists():
    !git clone https://github.com/codepranadev-svg/contentcreation.git /content/contentcreation
%cd /content/contentcreation
!pip -q install -r requirements.txt

# Point to the Drive episode folder created by Arena
DRIVE_EPISODE = Path('/content/drive/MyDrive/Rishton_Ke_Mod/2026-11-11')
CLIPS_DIR = DRIVE_EPISODE / '02_Live_Action_Cinema_Kit' / 'clips'
CLIPS_DIR.mkdir(parents=True, exist_ok=True)
print('Drive episode path:', DRIVE_EPISODE)
print('Drop live-action .mp4 clips named shot_01a.mp4 ... shot_05c.mp4 into:', CLIPS_DIR)

In [ ]:
# 2. Preview the 15-shot storyboard timings and cinema prompts
import json

ep_dir = Path('/content/contentcreation/hindi/2026-11-11')
script_data = json.loads((ep_dir / 'script.json').read_text(encoding='utf-8'))
timeline = json.loads((ep_dir / 'cue_timeline.json').read_text(encoding='utf-8'))

for sc in timeline['scenes']:
    print(f"Scene {sc['scene_number']}: {sc['scene_title']} ({sc['duration']:.1f}s)")
    for sh in sc.get('shots', []):
        stem = Path(sh['image']).stem
        mp4_candidate = CLIPS_DIR / f'{stem}.mp4'
        status = 'READY (.mp4 found)' if mp4_candidate.exists() else 'waiting for .mp4'
        print(f"  - {stem}: weight={sh['weight']} | {status}")

In [ ]:
# 3. Optional: Auto-swap any generated .mp4 clips from Drive into script.json and render 1080p
updated = 0
for seg in script_data['segments']:
    for sh in seg.get('shots', []):
        stem = Path(sh['image']).stem
        clip = CLIPS_DIR / f'{stem}.mp4'
        if clip.exists():
            dst = ep_dir / 'scenes' / f'{stem}.mp4'
            shutil.copy2(clip, dst)
            sh['image'] = f'scenes/{stem}.mp4'
            updated += 1
    # Also upgrade scene primary/short visual if clip exists
    for key in ('image', 'short_image'):
        if key in seg:
            stem = Path(seg[key]).stem
            if (ep_dir / 'scenes' / f'{stem}.mp4').exists():
                seg[key] = f'scenes/{stem}.mp4'

(ep_dir / 'script.json').write_text(json.dumps(script_data, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Swapped {updated} live-action .mp4 clips into script.json')

!python -m studio.daily_pipeline --episode-dir hindi/2026-11-11 --skip-audio --res 1080

# Copy finished 1080p renders back to Google Drive
out_drive = DRIVE_EPISODE / '03_Colab_1080p_Output'
out_drive.mkdir(parents=True, exist_ok=True)
for fname in (
    'baarish-ki-woh-aakhiri-raat_full_16x9.mp4',
    'baarish-ki-woh-aakhiri-raat_shorts_9x16.mp4',
    'thumbnail_16x9.jpg',
    'thumbnail_9x16.jpg',
    'subtitles.srt',
):
    src = ep_dir / fname
    if src.exists():
        shutil.copy2(src, out_drive / fname)
print('Uploaded outputs to Drive:', out_drive)